# Retrieval Augmented Generation (RAG) System for Policy Documents

In [ ]:
file_path = "/content/sample_data/vendor_policy.txt"
with open(file_path, 'r') as file:
    vendor_policy_text = file.read()

print("Successfully loaded the policy document")

## 1. Data Loading and Chunking

We start by loading the `vendor_policy.txt` document. To make it suitable for RAG, we apply a custom `section_based_chunking` function. This function aims to create semantic chunks based on numbered sections within the document. This approach helps ensure that each chunk contains a coherent piece of information.

In [ ]:
import re

def section_based_chunking(text):
    # Regex to find sections starting with '1. ', '2. ', '3. ', '4. ' etc.
    # It captures the section header and the content until the next header or end of text.
    sections = re.split(r'\n\n(\d+\.\s[A-Za-z]+\s[A-Za-z]+.*)', text)

    # The first element will be text before the first header, or empty if text starts with a header
    # We need to pair headers with their content.
    # Let's clean up and pair them
    chunks = []
    current_header = None
    for part in sections:
        if part.strip() == '':
            continue
        if re.match(r'^\d+\.\s[A-Za-z]+\s[A-Za-z]+.*', part):
            current_header = part.strip()
        else:
            if current_header:
                chunks.append(f"{current_header}\n{part.strip()}")
                current_header = None # Reset after appending
            else:
                # Handle any leading text that is not part of a numbered section if it exists
                chunks.append(part.strip())

    # Handle case where the text starts with a section header without any preceding text
    if not chunks and sections[0].strip() == '' and len(sections) > 1:
        # Re-process to correctly associate headers with their content when re.split creates an empty first element
        # This approach is a bit more robust:

        # Find all section headers and their starting positions
        matches = list(re.finditer(r'\n\n(\d+\.\s[A-Za-z]+\s[A-Za-z]+.*)', text))

        chunks = []
        start_idx = 0
        for i, match in enumerate(matches):
            # Content from the start_idx up to the current match's start
            if i == 0 and match.start() > 0:
                chunks.append(text[start_idx:match.start()].strip())

            # The section header and its content until the next header or end of text
            section_start = match.start(1) # Start of the captured group (the header itself)
            section_end = matches[i+1].start() if i + 1 < len(matches) else len(text)
            chunks.append(text[section_start:section_end].strip())




    return [chunk for chunk in chunks if chunk]

semantic_chunks = section_based_chunking(vendor_policy_text)

print(f"Total Semantic/Section-based Chunks: {len(semantic_chunks)}\n")

print("--- Semantic/Section-based Chunks ---")
for i, chunk in enumerate(semantic_chunks):
    print(f"--- Section {i+1} (length: {len(chunk)}) ---")
    print(chunk)
    print("\n")

### Observation on Chunking:

While the `section_based_chunking` successfully extracts most policy sections, it also produces a 'junk' chunk: `VENDOR PAYMENT POLICY`. This chunk represents the document's title and often gets retrieved due to its high-level relevance, even if it doesn't contain specific answers. This highlights a common challenge in chunking: ensuring all chunks are equally informative and free of noise.

## 2. Embedding Generation

Next, we use a `SentenceTransformer` model (`all-MiniLM-L6-v2`) to convert our text chunks into numerical vector representations (embeddings). These embeddings capture the semantic meaning of each chunk, allowing us to find similar chunks later on.

In [ ]:
from huggingface_hub import login
login(token="Your_API_Key")

In [ ]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
embedding_text = embedding_model.encode(semantic_chunks)

## 3. Query and Retrieval

With our chunks embedded, we can now process user queries. For each query, we generate its embedding and then calculate the cosine similarity between the query embedding and all chunk embeddings. The chunks with the highest similarity scores are considered the most relevant and are retrieved as context for the Language Model.

### Observation on Similarity Scores and Retrieval:

For `query1` ("What happens if my invoice doesn't have a PO number?"), the top chunk (`1. Invoice Submission`) has a high similarity score (0.5320), directly addressing the question. The second chunk (`2. Payment Terms`) has a significantly lower score (0.3158), indicating less direct relevance but still providing some contextual information. This demonstrates how the ranking helps prioritize the most pertinent information.



In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

query1 = "What happens if my invoice doesn't have a PO number?"
query1_embedding = embedding_model.encode([query1])[0]
#print(query1_embedding)

similarity_score_query1 = cosine_similarity(query1_embedding.reshape(1,-1), embedding_text)
#print(similarity_score_query1)
top_2_indices = np.argsort(similarity_score_query1.flatten())[::-1][:2]
top_2_chunks = [semantic_chunks[i] for i in top_2_indices]
top_2_scores = [similarity_score_query1.flatten()[i] for i in top_2_indices]

print("--- Top 2 chunks for the query ---")
for i, chunk in enumerate(top_2_chunks):
    print(f"--- Rank {i+1} (Score: {top_2_scores[i]:.4f}) ---")
    print(chunk)
    print("\n")

## Initialize GenerativeModel

### Subtask:
Initialize the `GenerativeModel` from the `google.generativeai` library, specifying the model (e.g., 'gemini-2.5-flash-preview-04-17') to be used for rephrasing.

**Reasoning**:
To use the Gemini API for content generation, we first need to initialize a `GenerativeModel` instance. We'll specify the model name, such as `'gemini-2.5-flash-preview-04-17'`, which is suitable for many generation tasks.

In [ ]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GeminiAPIKey1')
genai.configure(api_key=GOOGLE_API_KEY)

gemini_model = genai.GenerativeModel('gemini-flash-latest')

print("Gemini GenerativeModel initialized successfully.")

## Listing available Gemini models

To ensure we use a supported model, let's list the available Gemini models and check which ones support the `generateContent` method. This is important as some models might be deprecated or have specific capabilities.

In [ ]:
print('Available models that support generateContent:')
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

## Constructing the Prompt

### Subtask:
Define a prompt template that integrates the retrieved RAG context (`top_2_chunks`) and the user's question (`query1`) to guide the Gemini API in rephrasing the response.

## 4. Gemini API Integration for RAG (Generation)

Finally, we integrate the retrieved context with the user's question into a prompt for the Gemini API. The `gemini_model` then uses this combined information to generate a coherent and contextually accurate answer. This step leverages the LLM's generative capabilities, guided by the retrieved factual information, to provide a more nuanced and helpful response than simple keyword matching.

**Reasoning**:
To leverage the power of the RAG system, we need to provide the language model with both the user's original query and the relevant context (the `top_2_chunks`) we retrieved. This prompt template will structure that information for the Gemini model, ensuring it focuses on the provided context to answer the question.

In [ ]:
rag_prompt_template = """Based on the following context, answer the user's question. If the answer is not in the context, say so.

Context:
{retrieved_chunks}

Question: {user_query}"""

print("RAG prompt template defined successfully.")

## Prepare and Execute the Gemini API Call

### Subtask:
Fill the `rag_prompt_template` with the `top_2_chunks` and `query1`, and then make an API call to the `gemini_model` using the prepared prompt.

## 5. Testing with Additional Queries

To further validate our RAG system, we tested it with two more queries: "Can I get paid faster than Net 45?" and "Who handles unresolved payment disputes?".

**Reasoning**:
With the `rag_prompt_template` defined, we'll now combine the `top_2_chunks` (our retrieved context) and the `query1` (the user's question) into a single, comprehensive prompt string. This prompt will then be passed to the `gemini_model.generate_content()` method to generate an answer based on the provided RAG context.

In [ ]:
# Prepare the final prompt by filling in the template
formatted_prompt = rag_prompt_template.format(
    retrieved_chunks="\n\n".join(top_2_chunks), # Join chunks with double newlines for readability
    user_query=query1
)

print("--- Formatted Prompt for Gemini ---")
print(formatted_prompt)
print("\n")

# Make the API call to Gemini
response = gemini_model.generate_content(formatted_prompt)

print("--- Gemini's Rephrased Response ---")
print(response.text)

## Test with additional queries

In [ ]:
query2 = "Can I get paid faster than Net 45?"
query2_embedding = embedding_model.encode([query2])[0]

similarity_score_query2 = cosine_similarity(query2_embedding.reshape(1,-1), embedding_text)
top_2_indices_query2 = np.argsort(similarity_score_query2.flatten())[::-1][:2]
top_2_chunks_query2 = [semantic_chunks[i] for i in top_2_indices_query2]
top_2_scores_query2 = [similarity_score_query2.flatten()[i] for i in top_2_indices_query2]

print("--- Top 2 chunks for Query 2 ---")
for i, chunk in enumerate(top_2_chunks_query2):
    print(f"--- Rank {i+1} (Score: {top_2_scores_query2[i]:.4f}) ---")
    print(chunk)
    print("\n")

formatted_prompt_query2 = rag_prompt_template.format(
    retrieved_chunks="\n\n".join(top_2_chunks_query2),
    user_query=query2
)

print("--- Formatted Prompt for Gemini (Query 2) ---")
print(formatted_prompt_query2)
print("\n")

response_query2 = gemini_model.generate_content(formatted_prompt_query2)

print("--- Gemini's Rephrased Response (Query 2) ---")
print(response_query2.text)
print("\n" * 2)


### Observation on Query 2 Results:

For `query2`, the top chunk (`2. Payment Terms`) again had a strong similarity score (0.3847) and directly contained the answer. The 'junk' chunk (`VENDOR PAYMENT POLICY`) reappeared as the second-ranked chunk (0.1968), highlighting its persistent presence despite low specific relevance. The Gemini model successfully extracted the early payment discount and dynamic discounting program details from the relevant chunk.


In [ ]:
query3 = "Who handles unresolved payment disputes?"
query3_embedding = embedding_model.encode([query3])[0]

similarity_score_query3 = cosine_similarity(query3_embedding.reshape(1,-1), embedding_text)
top_2_indices_query3 = np.argsort(similarity_score_query3.flatten())[::-1][:2]
top_2_chunks_query3 = [semantic_chunks[i] for i in top_2_indices_query3]
top_2_scores_query3 = [similarity_score_query3.flatten()[i] for i in top_2_indices_query3]

print("--- Top 2 chunks for Query 3 ---")
for i, chunk in enumerate(top_2_chunks_query3):
    print(f"--- Rank {i+1} (Score: {top_2_scores_query3[i]:.4f}) ---")
    print(chunk)
    print("\n")

formatted_prompt_query3 = rag_prompt_template.format(
    retrieved_chunks="\n\n".join(top_2_chunks_query3),
    user_query=query3
)

print("--- Formatted Prompt for Gemini (Query 3) ---")
print(formatted_prompt_query3)
print("\n")

response_query3 = gemini_model.generate_content(formatted_prompt_query3)

print("--- Gemini's Rephrased Response (Query 3) ---")
print(response_query3.text)


### Observation on Query 3 Results:

For `query3`, the most relevant chunk (`3. Dispute Resolution`) had a very high similarity score (0.6721), leading to a precise answer from the Gemini model. The 'junk' chunk (`VENDOR PAYMENT POLICY`) was again the second result (0.4630), underscoring the potential for less specific but broadly related chunks to be retrieved. Despite this, the RAG system focused on the primary relevant chunk to provide the correct information.